In [1]:
from pyspark.sql import SparkSession

from sedona.spark import SedonaContext


config = (
    SedonaContext.builder()
    .config(
     "spark.driver.memory", "7G"   
    )
    .getOrCreate()
)

sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 18:17:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/07 18:17:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/07 18:17:22 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/07 18:17:22 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/07 18:17:22 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/07 18:17:22 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/12/07 18:17:22 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, 

In [2]:
geotiff_df = sedona\
    .read\
    .format("binaryFile")\
    .load("ID73_S20_W70_RP10_depth.tif")\
    .selectExpr("RS_FromGeoTiff(content) AS rast")\
    .selectExpr("RS_TileExplode(rast, 4096, 4096) AS (x, y, rast)")

In [3]:
row = geotiff_df.head()

In [4]:
type(row)

pyspark.sql.types.Row

In [5]:
raster = row.rast

In [6]:
type(raster)

sedona.spark.raster.sedona_raster.InDbSedonaRaster

In [7]:
raster.width

4096

In [8]:
raster.height

4096

In [9]:
raster.affine_trans.__dict__

{'scale_x': 0.0008333333356347339,
 'skew_y': 0.0,
 'skew_x': 0.0,
 'scale_y': -0.0008333333333325754,
 'ip_x': -70.00958226113656,
 'ip_y': -19.990433333333343,
 'pixel_anchor': <PixelAnchor.UPPER_LEFT: 2>}

In [10]:
print(raster.crs_wkt)

GEOGCS["WGS 84", 
  DATUM["World Geodetic System 1984", 
    SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], 
    AUTHORITY["EPSG","6326"]], 
  PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]], 
  UNIT["degree", 0.017453292519943295], 
  AXIS["Geodetic longitude", EAST], 
  AXIS["Geodetic latitude", NORTH], 
  AUTHORITY["EPSG","4326"]]


In [11]:
len(raster.bands_meta)

1

In [12]:
band = raster.bands_meta[0]

In [13]:
type(band)

sedona.spark.raster.meta.SampleDimension

In [14]:
band.__dict__

{'description': 'GRAY_INDEX', 'offset': 1.0, 'scale': 0.0, 'nodata': -9999.0}

In [15]:
{
    'description': 'GRAY_INDEX',
    'offset': 1.0,
    'scale': 0.0,
    'nodata': -9999.0
}

{'description': 'GRAY_INDEX', 'offset': 1.0, 'scale': 0.0, 'nodata': -9999.0}

In [16]:
type(raster.awt_raster)

sedona.spark.raster.awt_raster.AWTRaster

# Rasterio

In [17]:
rasterio_dataset = raster.as_rasterio()

In [18]:
type(rasterio_dataset)

rasterio.io.DatasetReader

In [19]:
rasterio_dataset.bounds

BoundingBox(left=-70.00958226113656, bottom=-23.40376666666357, right=-66.59624891837669, top=-19.990433333333343)